# Module 8 — Final Evaluation & Ship-It Decision
**Type:** [Watch Only]  
**Time:** 15 minutes  
**Job:** Assemble the master leaderboard. Make a defensible model selection recommendation. Save the final artifact.

---

> **Instructor note:** This module has no Code With Me cells. The code runs fast. The conversation is the point.
>
> Pacing:
> - 8.1–8.3: Setup, load artifacts, build master leaderboard (4 min)
> - 8.4–8.5: Final charts — wMAPE ranking, two-axis comparison (4 min)
> - 8.6: Ship-it framework (4 min) — **slow down here, this is the payoff**
> - 8.7: Save artifact, transition (3 min)

---
## 8.1 — Setup
**[Watch Only]**

> **Instructor note (30 sec):** Run silently.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

matplotlib.rcParams['figure.figsize'] = (14, 5)
matplotlib.rcParams['axes.spines.top'] = False
matplotlib.rcParams['axes.spines.right'] = False

from config import ARTIFACT_DIR, WORKSHOP_SUBSET_N, HORIZON, N_WINDOWS, MICRO_SUBSET_N
from src.checkpointing import load_checkpoint
from src.evaluation import score_forecasts, build_leaderboard

print("Setup complete.")

---
## 8.2 — Load All Scored Artifacts
**[Watch Only]**

> **Instructor note (1 min):** Load all forecast artifacts and score them in a single pass. This is the first time every model's scores are computed from the same call with the same subset_name — making the leaderboard directly comparable.

In [ ]:
# Load all full-subset forecast artifacts
baseline_full = load_checkpoint("04_baseline_forecasts")
ml_full       = load_checkpoint("05_ml_forecasts")
dl_full       = load_checkpoint("06_dl_forecasts")

# Score each stage — consistent subset_name across all
subset_label = f"workshop_{WORKSHOP_SUBSET_N}"
baseline_scores = score_forecasts(baseline_full, subset_name=subset_label)
ml_scores       = score_forecasts(ml_full,       subset_name=subset_label)
dl_scores       = score_forecasts(dl_full,       subset_name=subset_label)

# Build master leaderboard
master_lb = build_leaderboard([baseline_scores, ml_scores, dl_scores])

print(f"Master leaderboard: {len(master_lb)} models")
print(master_lb.to_string(index=False))

**Expected output:**
```
Master leaderboard: 6 models
          model stage         subset_name  Coverage_80  IntervalScore_80    wMAPE
       LightGBM    ml  workshop_1000        0.XX         XX.XXXX          0.XXXX
          NHITS    dl  workshop_1000        0.XX         XX.XXXX          0.XXXX
        AutoETS  base  workshop_1000        0.XX         XX.XXXX          0.XXXX
Chronos-t5-mini  base  workshop_1000        0.XX         XX.XXXX          0.XXXX
  SeasonalNaive  base  workshop_1000        0.XX         XX.XXXX          0.XXXX
          Naive  base  workshop_1000        0.XX         XX.XXXX          0.XXXX
```

---
## 8.3 — Final wMAPE Leaderboard Chart
**[Watch Only]**

> **Instructor note (2 min):** Show this chart and compare it to the destination preview from Module 1. The order may be different than the illustrative table — that is fine. Point out the gap between each tier: baseline → ML → DL.

In [ ]:
model_styles = {
    "Naive":           "#E53935",
    "SeasonalNaive":   "#1E88E5",
    "AutoETS":         "#43A047",
    "Chronos-t5-mini": "#FB8C00",
    "LightGBM":        "#7B1FA2",
    "NHITS":           "#F4511E",
}

if "wMAPE" in master_lb.columns:
    wmape_data = master_lb[["model", "wMAPE"]].dropna().sort_values("wMAPE")
    colors = [model_styles.get(m, "#90CAF9") for m in wmape_data["model"]]

    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.barh(wmape_data["model"], wmape_data["wMAPE"],
                   color=colors, edgecolor="white", height=0.6)
    ax.set_xlabel("wMAPE — pooled across 1,000 series × 3 windows × 28-day horizon")
    ax.set_title(
        f"Final Point Accuracy Leaderboard\n"
        f"M5 Workshop Subset — {WORKSHOP_SUBSET_N:,} series, {N_WINDOWS} CV windows, {HORIZON}-day horizon",
        fontsize=10
    )
    ax.invert_yaxis()
    for bar, val in zip(bars, wmape_data["wMAPE"]):
        ax.text(val + 0.001, bar.get_y() + bar.get_height() / 2,
                f"{val:.4f}", va="center", fontsize=9)

    # Stage tier annotations
    stage_map = {"Naive": "Baseline", "SeasonalNaive": "Baseline",
                 "AutoETS": "Baseline", "Chronos-t5-mini": "Baseline",
                 "LightGBM": "ML", "NHITS": "Deep Learning"}
    for bar, model in zip(bars, wmape_data["model"]):
        stage = stage_map.get(model, "")
        ax.text(0.001, bar.get_y() + bar.get_height() / 2,
                stage, va="center", fontsize=7, color="white", fontweight="bold")

    plt.tight_layout()
    plt.show()

**Expected output:** Full six-model horizontal bar chart sorted by wMAPE ascending. Stage labels (Baseline / ML / Deep Learning) embedded in each bar.

---
## 8.4 — Two-Axis Comparison: Accuracy and Uncertainty
**[Watch Only]**

> **Instructor note (2 min):** This chart makes the production decision legible. A model that dominates on both axes (bottom-left) is the clear choice. A model that wins wMAPE but has higher Interval Score requires a judgment call. Ask the room: "If these two dimensions conflict, which one matters more for your use case?"

In [ ]:
if "IntervalScore_80" in master_lb.columns and "wMAPE" in master_lb.columns:
    plot_data = master_lb.dropna(subset=["wMAPE", "IntervalScore_80"])

    fig, ax = plt.subplots(figsize=(9, 7))

    for _, row in plot_data.iterrows():
        color = model_styles.get(row["model"], "#555")
        ax.scatter(row["wMAPE"], row["IntervalScore_80"],
                   s=180, color=color, zorder=4, edgecolors="white", linewidth=1.5)
        ax.annotate(
            row["model"],
            xy=(row["wMAPE"], row["IntervalScore_80"]),
            xytext=(8, 3), textcoords="offset points",
            fontsize=9, color=color, fontweight="bold"
        )

    # Shade the "ideal" bottom-left quadrant
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()
    wmape_mid  = plot_data["wMAPE"].median()
    iscore_mid = plot_data["IntervalScore_80"].median()
    ax.axvline(wmape_mid,  color="#ccc", linestyle=":", linewidth=0.8)
    ax.axhline(iscore_mid, color="#ccc", linestyle=":", linewidth=0.8)
    ax.fill_between(
        [xlim[0], wmape_mid], ylim[0], iscore_mid,
        alpha=0.05, color="#43A047"
    )
    ax.text(xlim[0] + (wmape_mid - xlim[0]) * 0.1, ylim[0] + (iscore_mid - ylim[0]) * 0.08,
            "Production-ready zone", fontsize=8, color="#43A047", alpha=0.7)

    ax.set_xlabel("wMAPE — Point Forecast Accuracy (lower = better)", fontsize=10)
    ax.set_ylabel("Interval Score 80% — Uncertainty Quality (lower = better)", fontsize=10)
    ax.set_title("Final Model Comparison: Accuracy vs Uncertainty Quality", fontsize=11)
    plt.tight_layout()
    plt.show()

**Expected output:** Scatter plot with shaded "production-ready zone" in the bottom-left. Models in or near the zone are candidates for deployment.

---
## 8.5 — The Ship-It Framework
**[Watch Only]**

> **Instructor note (4 min):** Read through this section with the leaderboard on screen. Ask specific questions of the room — "LightGBM beats AutoETS by X wMAPE points. Is that worth maintaining a feature store?" Let them argue. There is no single right answer — the framework is the point.
>
> This is the session's payoff moment. Do not rush it.

You have a leaderboard. You have interval scores. Now make a decision.

**The question is not: which model has the best wMAPE?**  
The question is: **which model should go into production, given the operational constraints of your team?**

These are different questions. Use this framework:

---

### Step 1 — Eliminate models that fail the operational floor

Any model that cannot clear these thresholds is not a candidate regardless of accuracy:
- Can it retrain on a daily or weekly cadence within your compute budget?
- Can your team debug it when it fails in production?
- Does it have a clear owner who will maintain it?

Naive and SeasonalNaive pass all three. AutoETS passes. LightGBM and NHITS require infrastructure. Chronos requires GPU inference at scale.

---

### Step 2 — Quantify the accuracy improvement in business terms

A 3-point wMAPE improvement sounds meaningful. Connect it to a number:

- If your portfolio carries \$50M in inventory and you hold 2 weeks of safety stock,
  a 3-point wMAPE reduction translates to roughly \$1.5M in freed working capital.
- If your ML pipeline costs \$200K/year to run, maintain, and monitor — it pays for itself.
- If it costs \$2M — AutoETS is the better business decision even if LightGBM wins on accuracy.

**The leaderboard does not make this calculation. You do.**

---

### Step 3 — Check interval quality for your risk profile

If your procurement team uses forecast intervals to set safety stock:
- A model with better Interval Score reduces safety stock requirements directly
- A model that wins wMAPE but has erratic intervals cannot support that use case

Look at the scatter plot. If the wMAPE winner and the Interval Score winner are the same model, the decision is easy. If they differ, ask: which risk matters more — stockout risk (wMAPE) or overstocking risk (interval quality)?

---

### Step 4 — Make the call

A real recommendation sounds like this:

> *"Based on this evaluation, we recommend LightGBM for production. It delivers the best wMAPE and interval score on this portfolio, and the lag feature pipeline is manageable at our current scale. NHITS is not materially better to justify the GPU infrastructure. AutoETS remains the fallback if the feature pipeline degrades."*

That is a complete recommendation. It names a winner, justifies the choice, identifies the fallback, and draws a clear line at the infrastructure cost threshold.

---
## 8.6 — Save the Final Master Leaderboard
**[Watch Only]**

> **Instructor note (1 min):** Save as CSV — readable by anyone, no parquet tooling required. This is the artifact you hand to a stakeholder.

In [ ]:
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
leaderboard_path = ARTIFACT_DIR / "08_final_master_leaderboard.csv"
master_lb.to_csv(leaderboard_path, index=False)

print(f"  ✓ Final leaderboard saved: {leaderboard_path.name}")
print(f"    Models : {len(master_lb)}")
print(f"    Columns: {list(master_lb.columns)}")
print()
print("All workshop artifacts are complete:")

from src.checkpointing import list_checkpoints
list_checkpoints()

**Expected output:**
```
  ✓ Final leaderboard saved: 08_final_master_leaderboard.csv
    Models : 6
    Columns: ['model', 'stage', 'subset_name', 'Coverage_80', 'IntervalScore_80', 'wMAPE']

All workshop artifacts are complete:

  ARTIFACT REGISTRY STATUS
  ----------------------------------------
  ✓ EXISTS  00_env_status              ...
  ✓ EXISTS  02_global_config           ...
  ...
  ✓ EXISTS  08_final_master_leaderboard ...
```

---

> **Instructor note — transition:** "Every artifact is saved. The pipeline is complete. One module left — Q&A, next steps, and the companion asset."
>
> Open `instructor_notebooks/09_qa_and_next_steps.ipynb`.